In [1]:
import sys
import importlib.util
from pathlib import Path

import pandas as pd
import numpy as np

In [2]:
ROOT = Path.cwd() if (Path.cwd() / "Data").exists() else Path.cwd().parent


def _load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


resampling = _load_module("resampling", ROOT / "Data" / "resampled_data" / "resampling.py")

RAW_PATH = ROOT / "Data" / "raw_data" / "data_binance.parquet"
RESAMPLED_PATH = ROOT / "Data" / "resampled_data" / "resampled_data_4h.parquet"
CLEANED_PATH = ROOT / "Data" / "data_for_analysis" / "cleaned_data.parquet"

#### Load data

In [3]:
raw = pd.read_parquet(RAW_PATH)
resampled = pd.read_parquet(RESAMPLED_PATH)
cleaned = pd.read_parquet(CLEANED_PATH)

print(f"raw:       {len(raw):>7} rows | last timestamp = {raw['timestamp'].max()}")
print(f"resampled: {len(resampled):>7} rows | last timestamp = {resampled.index.max()}")
print(f"cleaned:   {len(cleaned):>7} rows | last timestamp = {cleaned.index.max()}")

raw:        588327 rows | last timestamp = 2026-09-06 12:55:00
resampled:   12262 rows | last timestamp = 2026-09-06 12:00:00
cleaned:     12262 rows | last timestamp = 2026-09-06 12:00:00


#### Freshness check — did the update actually pull new data?

In [4]:
now = pd.Timestamp.now("UTC").tz_localize(None)
staleness = now - raw["timestamp"].max()

assert staleness < pd.Timedelta(hours=6), \
    f"Validation Error: raw data looks stale — last candle is {staleness} old!"
print(f"✅ raw data is fresh (last candle {staleness} ago)")

✅ raw data is fresh (last candle 0 days 00:09:43.142131 ago)


#### Raw 5-min data — continuity & duplicates

In [ ]:
recent_cutoff = raw["timestamp"].max() - pd.Timedelta(days=2)
recent_raw = raw[raw["timestamp"] >= recent_cutoff]

gaps_recent = recent_raw["timestamp"].diff().dropna()
bad_gaps_recent = gaps_recent[gaps_recent != pd.Timedelta(minutes=5)]
assert bad_gaps_recent.empty, (
    f"Validation Error: {len(bad_gaps_recent)} gap(s) in recently updated 5-min data:\n{bad_gaps_recent}"
)
print("✅ recently updated 5-min data has no gaps")

✅ recently updated 5-min data has no gaps


#### Resampled 4h data — reuse the same strict validator as resampling.py

In [7]:
resampling.validate_rv_data(resampled)

✅ All checks passed: Data is clean, chronological, and ready for analysis.


#### Cleaned data — continuity + no unexpected NaNs in the freshly appended rows

In [8]:
assert cleaned.index.is_monotonic_increasing, "Validation Error: cleaned index not sorted"
assert cleaned.index.is_unique, "Validation Error: duplicate timestamps in cleaned data"

time_deltas = pd.Series(cleaned.index).diff().dropna()
bad_deltas = time_deltas[time_deltas != pd.Timedelta("4h")]
assert bad_deltas.empty, f"Validation Error: {len(bad_deltas)} gap(s) in cleaned 4h data"
print("✅ cleaned data has no gaps or duplicates")


✅ cleaned data has no gaps or duplicates


#### columns allowed to be NaN only in their "warm-up"/tail regions

In [10]:
expected_nan_cols = ["rv_daily", "rv_weekly", "rv_monthly", "vol_z_30d", "pump_risk", "log_rv_plus_t"]
other_cols = [c for c in cleaned.columns if c not in expected_nan_cols]

nan_in_other = cleaned[other_cols].isna().sum()
assert nan_in_other.sum() == 0, f"Validation Error: unexpected NaNs found:\n{nan_in_other[nan_in_other > 0]}"
print("✅ no unexpected NaNs outside of known warm-up/tail columns")


✅ no unexpected NaNs outside of known warm-up/tail columns


In [11]:
last_row = cleaned.iloc[-1]
unexpected_last_nan = last_row[other_cols].isna()
assert not unexpected_last_nan.any(), f"Validation Error: unexpected NaN in last row:\n{unexpected_last_nan[unexpected_last_nan]}"
print("✅ last row only has the expected NaN (log_rv_plus_t)")

# %% [markdown]
# #### Cross-file consistency — cleaned/resampled shouldn't drift apart

# %%
assert cleaned.index.max() == resampled.index.max(), (
    f"Validation Error: cleaned last date ({cleaned.index.max()}) != "
    f"resampled last date ({resampled.index.max()})"
)
print("✅ cleaned and resampled data are in sync")

print(f"\n🎉 All validations passed. Data is up to date through {cleaned.index.max()}.")

✅ last row only has the expected NaN (log_rv_plus_t)
✅ cleaned and resampled data are in sync

🎉 All validations passed. Data is up to date through 2026-09-06 12:00:00.
